In [4]:
import numpy as np
import cv2

1) Implement the full SVD from scratch without using power iteration. Instead, compute the eigendecomposition of 
A
T
A
A 
T
 A to get 
V
V and the singular values, then compute 
U
=
A
V
Σ
−
1
U=AVΣ 
−1
 . Compare numerical accuracy with your power iteration version and with NumPy.

In [ ]:
# Input matrix
A = np.array([
    [3, 1],
    [1, 3],
    [0, 2]
], dtype=float)

# Step 1: Calculate A transpose * A
ATA = np.dot(np.transpose(A),A)

# Step 2: Find eigenvalues and eigenvectors
eigenvalues, V = np.linalg.eigh(ATA)

# Step 3: Sort in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
V = V[:, idx]

# Step 4: Calculate singular values
S = np.sqrt(np.maximum(eigenvalues, 0))

# Step 5: Calculate U
U = np.dot(A,V)

U = U / S

# Step 6: Calculate V transpose
Vt = V.T

# Step 7: Reconstruct A
A_reconstructed = np.dot(np.dot(U, np.diag(S)), Vt)
# Step 8: Calculate error
error = np.linalg.norm(A - A_reconstructed)

# Step 9: Compare with NumPy SVD
U_np, S_np, Vt_np = np.linalg.svd(A, full_matrices=False)

np_error = np.linalg.norm(A - U_np @ np.diag(S_np) @ Vt_np)

print("Our Singular Values:", S)
print("NumPy Singular Values:", S_np)

print("\nOur Reconstruction Error:", error)
print("NumPy Reconstruction Error:", np_error)

print("\nReconstructed Matrix:")
print(A_reconstructed)

Our Singular Values: [4.28071902 2.38231918]
NumPy Singular Values: [4.28071902 2.38231918]

Our Reconstruction Error: 3.1401851738968297e-16
NumPy Reconstruction Error: 3.0204221995378595e-15

Reconstructed Matrix:
[[ 3.00000000e+00  1.00000000e+00]
 [ 1.00000000e+00  3.00000000e+00]
 [-1.26929067e-19  2.00000000e+00]]


2) Load a real grayscale image (or convert one to grayscale). Compress it at ranks 1, 5, 10, 25, 50, 100. For each rank, compute the compression ratio and the relative error. Find the rank where the image becomes visually acceptable.

In [ ]:

def image_to_grayscale(img_path):
    bgr_image = cv2.imread(img_path)
    gray_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2GRAY)
    return gray_image

def compress_image_svd(image_matrix, k):
    U, S, Vt = np.linalg.svd(image_matrix, full_matrices=False)
    compressed = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    return compressed



image = image_to_grayscale("riverside_rgb.png").astype(np.float64)
rows, cols = image.shape

for k in [1, 5, 10, 20, 50]:
    compressed = compress_image_svd(image, k)
    error = np.linalg.norm(image - compressed) / np.linalg.norm(image)
    original_size = rows * cols
    compressed_size = k * (rows + cols + 1)
    ratio = compressed_size / original_size
    print(f"k={k:>3d}  error={error:.4f}  storage={ratio:.1%}")

k=  1  error=0.3319  storage=0.2%
k=  5  error=0.2792  storage=1.1%
k= 10  error=0.2516  storage=2.3%
k= 20  error=0.2210  storage=4.6%
k= 50  error=0.1704  storage=11.4%


3) Build a tiny recommendation system. Create a 10x8 user-movie ratings matrix with some known entries. Fill missing entries with row means. Compute SVD and reconstruct a rank-3 approximation. Use the reconstructed matrix to predict the missing ratings. Verify that the predictions are reasonable.

In [11]:
import numpy as np

ratings = np.array([
    [5, 0, 0, 1, 0, 0, 2, 0],
    [4, 0, 0, 0, 5, 0, 0, 0],
    [0, 2, 4, 0, 0, 5, 0, 0],
    [1, 0, 5, 0, 0, 0, 2, 0],
    [0, 4, 0, 2, 0, 0, 5, 0],
    [3, 0, 2, 0, 4, 0, 0, 0],
    [0, 5, 0, 1, 0, 4, 0, 2],
    [2, 0, 0, 5, 0, 0, 4, 0],
    [4, 0, 1, 0, 0, 2, 5, 0],
    [0, 1, 0, 3, 5, 0, 0, 4]
], dtype=float)

filled = ratings.copy()

for i in range(len(ratings)):
    row_mean = np.mean(ratings[i][ratings[i] != 0])
    filled[i][filled[i] == 0] = row_mean

U, S, Vt = np.linalg.svd(filled, full_matrices=False)

U3 = U[:, :3]
S3 = np.diag(S[:3])
Vt3 = Vt[:3, :]

reconstructed = np.dot(np.dot(U3, S3), Vt3)

missing = ratings == 0

predictions = np.clip(reconstructed, 1, 5)

print("Filled Matrix:")
print(np.round(filled, 2))

print("\nRank-3 Reconstructed Matrix:")
print(np.round(reconstructed, 2))

print("\nPredicted Missing Ratings:")

for i, j in zip(*np.where(missing)):
    print(f"User {i+1}, Movie {j+1}: {predictions[i, j]:.2f}")

print("\nAll predictions between 1 and 5:")
print(np.all((predictions[missing] >= 1) & (predictions[missing] <= 5)))

Filled Matrix:
[[5.   2.67 2.67 1.   2.67 2.67 2.   2.67]
 [4.   4.5  4.5  4.5  5.   4.5  4.5  4.5 ]
 [3.67 2.   4.   3.67 3.67 5.   3.67 3.67]
 [1.   2.67 5.   2.67 2.67 2.67 2.   2.67]
 [3.67 4.   3.67 2.   3.67 3.67 5.   3.67]
 [3.   3.   2.   3.   4.   3.   3.   3.  ]
 [3.   5.   3.   1.   3.   4.   3.   2.  ]
 [2.   3.67 3.67 5.   3.67 3.67 4.   3.67]
 [4.   3.   1.   3.   3.   2.   5.   3.  ]
 [3.25 1.   3.25 3.   5.   3.25 3.25 4.  ]]

Rank-3 Reconstructed Matrix:
[[3.74 3.42 1.92 1.23 2.68 2.72 3.21 2.34]
 [4.03 4.13 4.68 4.19 4.95 4.74 4.7  4.5 ]
 [3.08 2.91 3.81 3.87 4.19 3.8  3.86 3.85]
 [0.8  2.76 4.51 2.81 2.72 3.33 1.88 2.51]
 [4.14 4.17 3.41 2.52 3.84 3.89 4.06 3.42]
 [3.21 2.55 2.51 2.78 3.41 2.96 3.47 3.1 ]
 [3.39 4.81 3.47 0.95 2.65 3.58 2.93 2.27]
 [2.51 2.76 4.26 4.14 4.18 3.88 3.59 3.86]
 [4.49 2.36 1.14 2.45 3.54 2.56 4.16 3.19]
 [2.94 1.78 2.81 4.02 4.03 3.1  3.77 3.74]]

Predicted Missing Ratings:
User 1, Movie 2: 3.42
User 1, Movie 3: 1.92
User 1, Movie 5: 2.68